# BÀI GIẢNG: PRETRAINED LANGUAGE MODELS — BERT, GPT, LLM HIỆN ĐẠI VÀ ỨNG DỤNG CHO TIẾNG VIỆT

**Học phần:** Deep Learning trong NLP · **Buổi trước:** Buổi 6 — Transformer · **Buổi sau:** Buổi 8 — LLM hiện đại
**Phiên bản:** 2.0 (nâng cấp chuyên sâu + chuyên đề tiếng Việt) · **Môi trường:** Google Colab (T4 GPU)

## Bản đồ bài giảng

| Phần | Nội dung | Thời lượng gợi ý |
|---|---|---|
| **A. Nền tảng** | Transfer learning, CLM/MLM, BERT, GPT, so sánh, fine-tuning, zero/few-shot | 90' |
| **B. Chuyên sâu** | Đếm tham số, causal mask, KV cache, GQA/RoPE, ModernBERT, scaling law, LoRA, DPO | 45' |
| **C. Tiếng Việt** | Đặc thù ngôn ngữ, tokenizer, tách từ, chuẩn hoá dấu, PhoBERT/ViSoBERT, NER, RAG tiếng Việt | 90' |
| **D. Thực hành gốc** | Fine-tune DistilBERT, GPT-2 decoding, BERT embedding | 30' |
| **E. Đánh giá** | Câu hỏi thảo luận, bài tập, tài liệu tham khảo | — |

> **Quy ước:** mỗi mục lớn mở đầu bằng khung **⏱ Tóm tắt 30 giây** — đọc khung này là nắm được ý chính; phần sau là giải thích sâu.

## Hướng dẫn chạy trên Colab
1. **Runtime → Change runtime type → T4 GPU**.
2. Chạy cell cài đặt. Nếu Colab báo cần restart, restart rồi chạy tiếp từ cell kiểm tra môi trường.
3. Các phần độc lập với nhau; phần không cần GPU: causal mask, LoRA mini, DPO mini, chuẩn hoá tiếng Việt, TF-IDF/BM25.
4. Code dùng API `transformers` mới (`eval_strategy`, `processing_class`) — tương thích transformers ≥ 4.46 và v5.

In [ ]:
# ============================================================
# CÀI ĐẶT MÔI TRƯỜNG
# ============================================================
!pip -q install -U transformers datasets evaluate accelerate peft sentencepiece \
    scikit-learn matplotlib seqeval sentence-transformers rank_bm25 underthesea pyvi

import sys, platform, math, random, time, re, unicodedata
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import transformers
print('Python      :', sys.version.split()[0])
print('PyTorch     :', torch.__version__)
print('Transformers:', transformers.__version__)
print('Device      :', device, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# PHẦN A — NỀN TẢNG

## I. MỤC TIÊU BÀI HỌC

**Kiến thức** — Sau buổi học, sinh viên có thể:
1. Giải thích vì sao *pre-training tự giám sát + fine-tuning* thay đổi căn bản cách làm NLP.
2. Phân biệt **Causal LM** và **Masked LM** về công thức, kiến trúc, loại bài toán phù hợp.
3. Trình bày BERT (MLM 80/10/10, NSP, WordPiece, [CLS]/[SEP]) và GPT (CLM, byte-level BPE, decoding).
4. Tính được số tham số, bộ nhớ KV cache; hiểu LoRA, DPO ở mức công thức.
5. Nhận diện **đặc thù tiếng Việt** (tách từ, dấu thanh, Unicode, tokenizer fertility) và chọn đúng mô hình tiếng Việt cho từng bài toán.

**Kỹ năng**
1. Fine-tune PhoBERT/DistilBERT cho phân loại và NER bằng HuggingFace (kể cả căn nhãn subword thủ công).
2. Xây pipeline tiền xử lý tiếng Việt và một hệ RAG tiếng Việt lai BM25 + dense.
3. Điều khiển sinh văn bản bằng greedy/beam/top-k/top-p/temperature.

**Thái độ** — Tư duy phản biện về bias, hallucination, chi phí tính toán; không mặc định mô hình tiếng Anh "dùng được" cho tiếng Việt.

---

## II. ĐỘNG LỰC CỦA TRANSFER LEARNING

> **⏱ Tóm tắt 30 giây.** Dữ liệu có nhãn thì ít, văn bản thô thì vô hạn. Tự tạo nhãn từ chính văn bản (che từ / đoán từ tiếp theo) để học ngôn ngữ nói chung, rồi tinh chỉnh bằng một ít dữ liệu có nhãn.

### 2.1 Vấn đề "đói dữ liệu" trước 2018
Pipeline cũ: thu thập 10K–100K câu có nhãn → dùng Word2Vec/GloVe (vector *tĩnh*, "bàn" chỉ có một vector dù là "cái bàn" hay "bàn bạc") → train từ đầu. Kết quả bị chặn trên bởi lượng nhãn.

### 2.2 Self-supervised learning
Văn bản tự chứa tín hiệu giám sát:
- *Che một từ, đoán lại* → phải hiểu ngữ pháp, ngữ nghĩa hai phía (MLM).
- *Đoán từ tiếp theo* → phải hiểu cách ngôn ngữ diễn tiến (CLM).

Ví dụ: "Hôm nay trời **[MASK]** rất đẹp và dễ chịu." → "nắng"/"trong" hợp lý hơn "mưa" vì cụm "đẹp và dễ chịu". Lặp lại trên hàng tỷ câu → biểu diễn ngôn ngữ phong phú **không cần nhãn**.

### 2.3 Hai giai đoạn

| | Pre-training | Fine-tuning |
|---|---|---|
| Dữ liệu | Hàng chục GB → hàng chục TB văn bản thô | Vài trăm → vài chục nghìn mẫu có nhãn |
| Mục tiêu | MLM / CLM (tự giám sát) | Loss của bài toán (CE, span, …) |
| Chi phí | Tuần–tháng, nhiều GPU/TPU | Phút–giờ, 1 GPU |
| Tần suất | Một lần, dùng nhiều lần | Mỗi bài toán một lần |

**Vì sao hiệu quả?** Tầng thấp học đặc trưng chung (hình thái, cú pháp), tầng cao học đặc trưng gần nhiệm vụ. Fine-tuning chủ yếu "uốn" tầng cao, không học lại từ đầu.

### 2.4 Dòng thời gian
- **2013–2017 — feature-based:** Word2Vec, GloVe, FastText (tĩnh); **ELMo (2018)**: BiLSTM, vector theo ngữ cảnh nhưng dùng như đặc trưng cố định.
- **2018 — fine-tuning toàn mô hình:** GPT-1 (6/2018), BERT (10/2018).
- **2019–2020 — scaling & cải tiến:** RoBERTa, ALBERT, DistilBERT, ELECTRA, T5, BART, GPT-2, GPT-3.
- **2022 → nay — LLM căn chỉnh:** InstructGPT/ChatGPT (RLHF), DPO, LLM mã nguồn mở (LLaMA, Mistral, Qwen, Gemma), mô hình suy luận, encoder thế hệ mới (ModernBERT).

---

## III. LANGUAGE MODELING — CLM, MLM VÀ CÁC BIẾN THỂ

> **⏱ Tóm tắt 30 giây.** CLM đoán token tiếp theo, chỉ nhìn bên trái → sinh văn bản tự nhiên. MLM đoán token bị che, nhìn hai phía → hiểu văn bản tốt. Perplexity = "số lựa chọn trung bình mô hình phân vân" — càng thấp càng tốt.

### 3.1 Chain rule và perplexity
$$P(w_1,\dots,w_T)=\prod_{t=1}^{T}P(w_t\mid w_{<t}),\qquad \text{PPL}=\exp\Big(-\tfrac{1}{T}\sum_{t}\log P(w_t\mid w_{<t})\Big)$$
- Mô hình hoàn hảo: PPL = 1. Mô hình đoán đều trên $V$ token: PPL = $V$.
- ⚠️ **PPL chỉ so sánh được giữa các mô hình dùng *cùng tokenizer*.** Tokenizer chia nhỏ hơn → nhiều token dễ đoán hơn → PPL/token thấp giả tạo. Khi so sánh chéo, dùng *bits-per-character* hoặc PPL theo từ/âm tiết.

### 3.2 Causal LM (GPT)
$$\mathcal{L}_{\text{CLM}}=-\sum_{t=1}^{T}\log P_\theta(w_t\mid w_{<t})$$
Dùng **causal mask** (ma trận tam giác) để vị trí $t$ không nhìn thấy $t+1,\dots,T$. Loss tính ở **mọi** vị trí → hiệu quả mẫu cao.

### 3.3 Masked LM (BERT)
$$\mathcal{L}_{\text{MLM}}=-\frac{1}{|\mathcal M|}\sum_{i\in\mathcal M}\log P_\theta(w_i\mid\tilde{\mathbf w})$$
Chỉ tính loss ở ~15% vị trí được chọn $\mathcal M$; bù lại mỗi dự đoán thấy ngữ cảnh **hai chiều**.

### 3.4 Minh hoạ trực quan
```
CLM (GPT):  "Tôi yêu học máy"
  P(yêu | Tôi) → P(học | Tôi yêu) → P(máy | Tôi yêu học)     ← chỉ nhìn trái
MLM (BERT): "Tôi yêu [MASK] máy học sâu"
  h[MASK] thấy cả "Tôi yêu" và "máy học sâu" → đoán "học"      ← nhìn hai phía
```

### 3.5 Bức tranh đầy đủ các mục tiêu pre-training

| Mục tiêu | Mô hình | Ý tưởng | Điểm mạnh |
|---|---|---|---|
| CLM | GPT, LLaMA, Qwen | Đoán token kế tiếp | Sinh văn bản, ICL |
| MLM | BERT, RoBERTa, PhoBERT | Đoán token bị che | Hiểu, phân loại, NER |
| RTD (Replaced Token Detection) | ELECTRA, ViDeBERTa* | Phân biệt token thật/bị thay | Hiệu quả mẫu cao (loss ở 100% vị trí) |
| Span corruption | T5, ViT5 | Che cả đoạn, sinh lại đoạn | Seq2seq: tóm tắt, dịch |
| Denoising | BART, BARTpho | Làm hỏng câu (xoá, đảo), khôi phục | Sinh có điều kiện |

\* ViDeBERTa dùng DeBERTaV3 với mục tiêu RTD.

---

## IV. BERT — BIDIRECTIONAL ENCODER REPRESENTATIONS FROM TRANSFORMERS

> **⏱ Tóm tắt 30 giây.** BERT = chồng Encoder Transformer, học bằng MLM (+NSP). Mỗi token nhìn toàn câu. Muốn làm bài toán gì thì gắn một "đầu" nhỏ lên trên rồi fine-tune cả mô hình.

### 4.1 Kiến trúc

| | BERT-Base | BERT-Large |
|---|---|---|
| Layers $L$ | 12 | 24 |
| Hidden $d$ | 768 | 1024 |
| Heads | 12 | 16 |
| FFN $d_{ff}$ | 3072 | 4096 |
| Tham số | ~110M | ~340M |
| Pre-training | 4 ngày, 16 TPU chip (4 Cloud TPU) | 4 ngày, 64 TPU chip (16 Cloud TPU) |

### 4.2 Đếm tham số — tự tính được, không cần tra
Một Encoder layer (có bias):
- Multi-Head Attention: $W_Q,W_K,W_V,W_O$ → $4(d^2+d)$
- FFN: $d\cdot d_{ff}+d_{ff}+d_{ff}\cdot d+d$
- 2 LayerNorm: $2\times 2d$

Với BERT-Base: MHA = 2,362,368; FFN = 4,722,432; LN = 3,072 → **7,087,872 / layer** → ×12 = **85.05M**.
Embedding: $30522\times768$ (token) + $512\times768$ (vị trí) + $2\times768$ (segment) + LN = 23.84M; pooler 0.59M → **≈109.5M**. (Cell kiểm chứng ở Phần B.)

> 💡 Quy tắc nhanh: mỗi layer ≈ $12d^2$ tham số (vì $d_{ff}=4d$: $4d^2 + 8d^2$).

### 4.3 MLM với chiến lược 80/10/10
Chọn 15% token; trong số đó: **80%** → `[MASK]`, **10%** → token ngẫu nhiên, **10%** → giữ nguyên.
- *Vì sao không 100% [MASK]?* `[MASK]` không bao giờ xuất hiện khi fine-tune → lệch train/inference.
- *10% ngẫu nhiên:* buộc mô hình kiểm tra lại **mọi** token bằng ngữ cảnh (không tin mù token hiện tại).
- *10% giữ nguyên:* buộc biểu diễn token thật cũng phải "dự đoán được" → biểu diễn ổn định.

Nhược điểm: chỉ 15% vị trí có loss → hội tụ chậm hơn CLM; ELECTRA khắc phục bằng RTD.

### 4.4 Next Sentence Prediction (NSP)
Cặp (A, B), 50% B là câu tiếp theo, 50% ngẫu nhiên; phân loại từ $\mathbf h_{[CLS]}$. $\mathcal L_{\text{BERT}}=\mathcal L_{\text{MLM}}+\mathcal L_{\text{NSP}}$.
**RoBERTa (2019)** cho thấy bỏ NSP, train lâu hơn, dữ liệu nhiều hơn, dynamic masking → tốt hơn. PhoBERT theo công thức RoBERTa nên **không có NSP**.

### 4.5 WordPiece
Bắt đầu từ ký tự, lặp lại việc gộp cặp làm **tăng likelihood** của corpus nhiều nhất (tiêu chí: $\frac{\text{count}(ab)}{\text{count}(a)\,\text{count}(b)}$), dừng ở ~30K token. Tiền tố `##` = mảnh không đứng đầu từ: `playing → play ##ing`.

### 4.6 Biểu diễn đầu vào
`Input = Token Emb + Segment Emb + Position Emb` (position **học được**, tối đa 512).
- `[CLS]`: đầu chuỗi, biểu diễn tổng hợp cho phân loại.
- `[SEP]`: ranh giới câu A/B.

*Vì sao [CLS] "biết" cả câu dù không bị mask?* Qua self-attention nó được trộn thông tin từ mọi token ở mọi tầng; NSP (và sau đó loss fine-tune) buộc nó mã hoá thông tin toàn cục.

### 4.7 Fine-tuning cho 4 dạng bài toán
| Bài toán | Đầu vào | Đầu (head) | Loss |
|---|---|---|---|
| Phân loại câu | `[CLS] x [SEP]` | Linear trên $h_{[CLS]}$ | CE |
| Gán nhãn token (NER, POS) | `[CLS] w1..wT [SEP]` | Linear trên từng $h_i$ | CE từng vị trí (bỏ qua subword phụ: nhãn −100) |
| QA trích xuất | `[CLS] q [SEP] passage [SEP]` | 2 vector $w_{start}, w_{end}$ | CE(start)+CE(end) |
| Cặp câu (NLI) | `[CLS] premise [SEP] hypothesis [SEP]` | Linear trên $h_{[CLS]}$ | CE 3 lớp |

**Siêu tham số điển hình:** LR 2e-5→5e-5, batch 16–32, 2–4 epoch, warmup 6–10%, weight decay 0.01.
Thành tích lịch sử: BERT-Large đạt **90.9 F1** trên SQuAD 1.1 (đơn mô hình); bản ensemble có thêm dữ liệu TriviaQA đạt 93.2 F1, vượt mức người (91.2).

### 4.8 🆕 Encoder thế hệ mới: ModernBERT (12/2024)
BERT gốc bị giới hạn 512 token và kiến trúc 2018. ModernBERT mang các cải tiến từ LLM vào encoder:
- **RoPE** thay position embedding học được → context **8.192 token**.
- **Attention xen kẽ cục bộ/toàn cục** (sliding window 128 và global mỗi 3 layer) → nhanh với chuỗi dài.
- **Unpadding + FlashAttention**, GeGLU, bỏ bias, pre-training ~2T token có cả code.

→ Encoder vẫn là lựa chọn số 1 cho **retrieval, reranking, phân loại, NER** vì rẻ và nhanh hơn LLM hàng chục lần. (ModernBERT gốc chủ yếu tiếng Anh; với tiếng Việt xem Phần C.)

---

## V. GPT — GENERATIVE PRETRAINED TRANSFORMER

> **⏱ Tóm tắt 30 giây.** GPT = chồng Decoder có causal mask, học đoán token kế tiếp. Sinh văn bản bằng cách lặp: dự đoán → chọn token → nối vào → lặp. Chất lượng đầu ra phụ thuộc mạnh vào cách **chọn** token (decoding).

### 5.1 Kiến trúc decoder-only
Mỗi layer: `Masked Self-Attention → Add&Norm → FFN → Add&Norm` (không có cross-attention). GPT-2 trở đi dùng **Pre-LN** (LayerNorm trước sub-layer) để train sâu ổn định hơn; LLM hiện đại dùng thêm RMSNorm, SwiGLU, RoPE, GQA.

| | GPT-1 | GPT-2 (XL) | GPT-3 | GPT-4 |
|---|---|---|---|---|
| Năm | 2018 | 2019 | 2020 | 2023 |
| Layers | 12 | 48 | 96 | không công bố |
| Hidden | 768 | 1600 | 12288 | không công bố |
| Tham số | 117M | 1.5B | 175B | không công bố |
| Dữ liệu | BooksCorpus | WebText 40GB | ~300B token | không công bố |
| Context | 512 | 1024 | 2048 | 8K → 128K (Turbo) |

### 5.2 Vì sao CLM hợp với decoder?
Causal mask đảm bảo khi dự đoán $w_t$ mô hình không "nhìn trộm" $w_t$. Ưu điểm: loss ở 100% vị trí (≈ 6.7 lần nhiều tín hiệu hơn MLM 15%), khớp tự nhiên với việc sinh, một mục tiêu duy nhất.

### 5.3 Chiến lược giải mã (decoding)
| Chiến lược | Cách làm | Khi nào dùng |
|---|---|---|
| Greedy | $\arg\max$ mỗi bước | Câu trả lời ngắn, xác định; dễ lặp |
| Beam search ($k$) | Giữ $k$ chuỗi tốt nhất | Dịch máy, tóm tắt; ít sáng tạo |
| Temperature $T$ | $\text{softmax}(z/T)$ | $T<1$ an toàn, $T>1$ đa dạng |
| Top-k | Chỉ lấy mẫu trong $k$ token cao nhất | Cắt đuôi phân phối cố định |
| Top-p (nucleus) | Tập nhỏ nhất có tổng xác suất ≥ $p$ | Mặc định phổ biến ($p$=0.9) |
| Min-p | Giữ token có $p \ge p_{min}\cdot p_{max}$ | Thích nghi tốt ở nhiệt độ cao |
| Repetition penalty / no_repeat_ngram | Phạt token đã xuất hiện | Chống lặp |

**Top-p hơn top-k ở đâu?** Phân phối nhọn → nucleus nhỏ (ít lựa chọn tệ); phân phối bẹt → nucleus lớn (đủ đa dạng). Top-k không thích nghi được.

### 5.4 Byte-level BPE
BPE gộp cặp **xuất hiện nhiều nhất** (khác WordPiece: tối đa likelihood). GPT-2 làm trên **byte UTF-8** → không bao giờ OOV, vocab 50.257.
⚠️ Mặt trái với tiếng Việt: mỗi ký tự có dấu ("ệ", "ữ") chiếm 2–3 byte, và từ vựng GPT-2 gần như không có âm tiết tiếng Việt → một âm tiết bị chẻ thành 3–6 token. Tokenizer của LLM đa ngữ mới (Qwen, Llama 3, Gemma) tốt hơn nhiều — xem đo đạc ở Phần C.

### 5.5 🆕 KV cache — vì sao sinh văn bản nhanh được
Khi sinh token thứ $t$, Key/Value của $t-1$ token trước **không đổi** → lưu lại, mỗi bước chỉ tính Q/K/V cho token mới. Chi phí mỗi bước giảm từ tính lại toàn chuỗi xuống $O(t)$ cho attention.
Giá phải trả là bộ nhớ:
$$\text{KV bytes/token} = 2\times L\times n_{kv}\times d_{head}\times \text{bytes}$$
Ví dụ Qwen2.5-7B ($L$=28, $n_{kv}$=4 nhờ **GQA**, $d_{head}$=128, fp16): $2\cdot28\cdot4\cdot128\cdot2$ = **56 KB/token** → 32K token ≈ **1,8 GB**. Nếu không có GQA (28 head KV) con số này gấp 7 lần.

### 5.6 Fine-tuning GPT và In-Context Learning
- GPT-1: head tuyến tính trên token cuối.
- GPT-3+: **prompting** — zero-shot (chỉ mô tả), few-shot (vài ví dụ trong prompt, không cập nhật trọng số).
```
Phân loại cảm xúc:
"Phim hay quá!" → tích cực
"Chán ngắt."   → tiêu cực
"Không phải là phim dở." →
```

---

## VI. SO SÁNH BA HỌ KIẾN TRÚC

> **⏱ Tóm tắt 30 giây.** Hiểu → encoder. Sinh tự do → decoder. Biến đổi chuỗi sang chuỗi → encoder-decoder. Hệ thống thực tế thường **kết hợp**.

| Tiêu chí | Encoder-only (BERT) | Decoder-only (GPT) | Encoder-Decoder (T5/BART) |
|---|---|---|---|
| Mục tiêu | MLM (+NSP) | CLM | Span corruption / denoising |
| Ngữ cảnh | Hai chiều | Một chiều (trái) | Hai chiều (enc) + một chiều (dec) |
| Tokenizer | WordPiece / BPE | Byte-level BPE | SentencePiece |
| Suy luận | 1 lượt forward, $O(T^2)$ | Tuần tự $T$ bước, có KV cache | Encode 1 lần + decode tuần tự |
| Mạnh ở | Phân loại, NER, QA trích xuất, embedding, rerank | Hội thoại, code, ICL, agent | Tóm tắt, dịch, sửa lỗi, khôi phục dấu |
| Zero/few-shot | Kém | Mạnh | Trung bình (Flan-T5 khá) |
| Chi phí | Thấp nhất | Cao nhất | Trung bình |
| Đại diện tiếng Việt | PhoBERT, ViSoBERT, ViDeBERTa | Vistral, SeaLLM, Qwen2.5 (đa ngữ) | ViT5, BARTpho |

**Chọn nhanh:** phân loại/NER dữ liệu có nhãn → encoder; không có nhãn, cần linh hoạt → LLM prompting; tóm tắt/dịch có dữ liệu cặp → encoder-decoder; hỏi đáp trên tài liệu → **RAG** (encoder truy hồi + decoder sinh).

---

## VII. FINE-TUNING — TỪ FULL FINE-TUNING ĐẾN PEFT

> **⏱ Tóm tắt 30 giây.** Fine-tune toàn bộ tốt nhưng đắt. LoRA chỉ học một cập nhật hạng thấp $BA$ (thường <1% tham số) mà chất lượng gần tương đương.

### 7.1 Kỹ thuật ổn định khi fine-tune
- **Warmup + linear/cosine decay:** tránh gradient lớn đầu tiên phá kiến thức pre-train (*catastrophic forgetting*).
- **Layer-wise LR decay:** tầng thấp LR nhỏ, tầng cao LR lớn.
- **Nhiều seed:** fine-tune BERT trên dữ liệu nhỏ dao động mạnh theo seed → báo cáo trung bình ± độ lệch chuẩn của ≥3 seed.

### 7.2 NER và vấn đề subword
Một từ → nhiều subword. Chuẩn: gán nhãn cho **subword đầu**, các subword sau nhận nhãn **−100** (bị `CrossEntropyLoss` bỏ qua). Khi dự đoán, lấy nhãn tại subword đầu.

### 7.3 Sentence-BERT (SBERT)
Vector [CLS] thô không tốt cho cosine similarity. SBERT fine-tune kiểu siamese (NLI, contrastive) + mean pooling → embedding câu dùng cho semantic search. Các mô hình embedding hiện đại (E5, BGE-M3) là hậu duệ, huấn luyện contrastive với **in-batch negatives**:
$$\mathcal L=-\log\frac{e^{\cos(q,p^+)/\tau}}{\sum_{j}e^{\cos(q,p_j)/\tau}}$$

### 7.4 🆕 LoRA
$$W' = W_0 + \tfrac{\alpha}{r}BA,\quad A\in\mathbb R^{r\times d_{in}},\ B\in\mathbb R^{d_{out}\times r},\ r\ll d$$
- $W_0$ đóng băng; $B$ khởi tạo **0** → ban đầu mô hình y hệt bản gốc.
- Tham số học thêm mỗi ma trận: $r(d_{in}+d_{out})$. Ví dụ $768\times768$, $r$=8: 12.288 thay vì 589.824 (**2,1%**).
- Sau khi train có thể **gộp** $BA$ vào $W_0$ → không tăng độ trễ suy luận.
- **QLoRA:** lượng tử hoá $W_0$ về 4-bit (NF4) + LoRA → fine-tune LLM 7B trên 1 GPU 16GB.

---

## VIII. ZERO-SHOT, FEW-SHOT, INSTRUCTION TUNING VÀ CĂN CHỈNH

> **⏱ Tóm tắt 30 giây.** LLM đủ lớn "học" từ ví dụ trong prompt mà không đổi trọng số (ICL). Để thành trợ lý hữu ích: SFT trên cặp (chỉ dẫn, trả lời) → căn chỉnh theo sở thích con người (RLHF hoặc DPO).

### 8.1 In-Context Learning
- **Zero-shot:** chỉ mô tả nhiệm vụ. **Few-shot:** thêm 1–32 ví dụ.
- ICL không cập nhật tham số; nhiều nghiên cứu diễn giải attention như thực hiện một dạng "gradient descent ngầm" trên ví dụ trong ngữ cảnh — vẫn là câu hỏi mở.
- **Chain-of-Thought:** yêu cầu/ví dụ suy luận từng bước → cải thiện mạnh bài toán nhiều bước ở mô hình lớn.
- ICL nhạy với thứ tự ví dụ, cách đặt nhãn, định dạng → luôn thử vài biến thể prompt.

### 8.2 Pipeline căn chỉnh
```
Pre-trained LM ──SFT (instruction data)──► SFT model ──┬─ RLHF: train Reward Model → PPO (+ phạt KL)
                                                        └─ DPO : tối ưu trực tiếp trên cặp (chosen, rejected)
```
**DPO:**
$$\mathcal L_{DPO}=-\log\sigma\Big(\beta\Big[\log\tfrac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)}-\log\tfrac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\Big]\Big)$$
Ý nghĩa: tăng xác suất tương đối của câu được chọn $y_w$ so với câu bị loại $y_l$, **so với mô hình tham chiếu**; $\beta$ lớn → bám sát tham chiếu hơn. Không cần reward model riêng, không cần RL → đơn giản, ổn định.

---

## IX. HẠN CHẾ CỦA PRETRAINED LANGUAGE MODELS

> **⏱ Tóm tắt 30 giây.** Đắt, thiên lệch, bịa đặt tự tin, giới hạn ngữ cảnh, kiến thức đóng băng. Với tiếng Việt còn thêm: dữ liệu ít hơn, tokenizer kém hiệu quả, đánh giá thiếu chuẩn.

**9.1 Chi phí.** BERT-Large vài nghìn USD (2018); GPT-3 ước tính vài triệu USD (2020). Chi phí suy luận nhân với hàng triệu yêu cầu/ngày thường vượt chi phí huấn luyện. Hệ quả: bất bình đẳng tiếp cận nghiên cứu, dấu chân carbon.

**9.2 Bias.** Mô hình phản ánh định kiến trong dữ liệu (nghề nghiệp–giới tính, vùng miền…).
*Đo lường:* WEAT/SEAT (liên kết embedding), StereoSet, CrowS-Pairs, probing; với tiếng Việt thường phải tự xây bộ kiểm thử (ví dụ hoán đổi tên nam/nữ, Bắc/Trung/Nam).
*Giảm thiểu:* lọc/cân bằng dữ liệu, counterfactual data augmentation, căn chỉnh (RLHF/DPO), đánh giá theo nhóm.

**9.3 Hallucination.** Mục tiêu CLM thưởng cho "nghe hợp lý", không thưởng cho "đúng sự thật" → sinh trích dẫn giả, số liệu sai với độ tự tin cao.
*Giảm thiểu:* RAG + bắt buộc trích nguồn, cho phép trả lời "không biết", tự kiểm chứng, nhiệt độ thấp cho câu hỏi sự kiện. Không thể loại bỏ hoàn toàn vì mô hình là bộ nén có mất mát của dữ liệu.

**9.4 Ngữ cảnh.** Attention $O(T^2)$. Giải pháp: FlashAttention (giảm bộ nhớ, không đổi độ phức tạp), sliding window, RoPE scaling (YaRN), Ring Attention. Context dài ≠ dùng tốt: hiện tượng *lost in the middle* — thông tin ở giữa ngữ cảnh dài dễ bị bỏ sót.

| Mô hình | Context |
|---|---|
| BERT / PhoBERT | 512 / **256** token |
| GPT-2 | 1.024 |
| GPT-3 | 2.048 |
| ModernBERT | 8.192 |
| LLM thương mại 2024–2026 | 128K → 1M+ |

**9.5 Khác.** Knowledge cutoff; thiếu grounding đa giác quan (trừ mô hình đa phương thức); suy luận nhiều bước và nhân quả còn dễ vỡ.

---

## X. 🆕 SCALING LAWS — VÌ SAO "TO HƠN" LẠI "GIỎI HƠN"

> **⏱ Tóm tắt 30 giây.** Loss giảm theo luỹ thừa khi tăng tham số $N$, dữ liệu $D$, tính toán $C$. Với ngân sách cố định, nên tăng $N$ và $D$ **cân đối**: ~20 token cho mỗi tham số (Chinchilla).

- **Kaplan et al. (2020):** $\text{Loss}\propto N^{-0.076}$, $\text{Loss}\propto D^{-0.095}$ — luật luỹ thừa trải qua nhiều bậc độ lớn.
- **Chinchilla (Hoffmann et al., 2022):** tối ưu tính toán khi $D\approx 20N$; Chinchilla 70B/1.4T token vượt Gopher 280B/300B token. GPT-3 (175B/300B token) là "thiếu dữ liệu".
- **Thực tế 2024–2026:** mô hình nhỏ được train **vượt xa** tỉ lệ 20:1 (hàng nghìn tỉ token cho mô hình 1–8B) vì chi phí *suy luận* mới là chi phí lớn nhất → mô hình nhỏ mà giỏi.
- **Test-time compute:** mô hình suy luận (reasoning models) cho thấy tăng tính toán **lúc suy luận** (nghĩ dài hơn) cũng là một trục scaling mới.

---

# PHẦN B — CHUYÊN SÂU CÓ CODE

Các cell dưới đây minh hoạ trực tiếp các khái niệm ở Phần A. Phần lớn chạy được trên CPU.

In [ ]:
# ============================================================
# B1. KIỂM CHỨNG CÔNG THỨC ĐẾM THAM SỐ BERT-BASE
# ============================================================
from transformers import BertConfig, BertModel

def encoder_layer_params(d, d_ff):
    mha = 4 * (d * d + d)
    ffn = d * d_ff + d_ff + d_ff * d + d
    ln  = 2 * (2 * d)
    return mha, ffn, ln, mha + ffn + ln

mha, ffn, ln, per_layer = encoder_layer_params(768, 3072)
emb = 30522 * 768 + 512 * 768 + 2 * 768 + 2 * 768
pooler = 768 * 768 + 768
print(f'MHA={mha:,}  FFN={ffn:,}  LN={ln:,}  → 1 layer = {per_layer:,}')
print(f'12 layers = {12*per_layer:,} | embedding = {emb:,} | pooler = {pooler:,}')
print(f'Tổng theo công thức: {12*per_layer + emb + pooler:,}')

# Đối chiếu với mô hình thật (khởi tạo ngẫu nhiên, không cần tải trọng số)
real = BertModel(BertConfig())
print(f'Tổng thực tế (HF)  : {sum(p.numel() for p in real.parameters()):,}')

# Bài tập nhanh: tính cho GPT-2 Medium (d=1024, d_ff=4096, 24 layers)
print(f'GPT-2 Medium, 1 decoder layer ≈ {encoder_layer_params(1024, 4096)[3]:,} tham số')
del real

In [ ]:
# ============================================================
# B2. CAUSAL MASK — GPT "KHÔNG ĐƯỢC NHÌN TƯƠNG LAI"
# ============================================================
import torch, torch.nn.functional as F

T, d = 5, 8
torch.manual_seed(0)
Q, K = torch.randn(T, d), torch.randn(T, d)
scores = Q @ K.T / d**0.5

causal_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)  # True = bị chặn
attn_bidir  = F.softmax(scores, dim=-1)
attn_causal = F.softmax(scores.masked_fill(causal_mask, float('-inf')), dim=-1)

torch.set_printoptions(precision=2, sci_mode=False)
print('Attention hai chiều (BERT):\n', attn_bidir)
print('\nAttention nhân quả (GPT) — phần trên đường chéo = 0:\n', attn_causal)

In [ ]:
# ============================================================
# B3. MASKING 80/10/10 THỰC TẾ BẰNG DataCollatorForLanguageModeling
# ============================================================
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

tok = AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')
collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=0.15)

text = 'Sinh viên Đại học Thủy lợi nghiên cứu mô hình ngôn ngữ cho tiếng Việt và ứng dụng trong quản lý tài nguyên nước.'
enc = tok(text)
torch.manual_seed(1)
batch = collator([enc] * 3)   # 3 lần masking ngẫu nhiên khác nhau
orig = tok.convert_ids_to_tokens(enc['input_ids'])
for k in range(3):
    ids, labels = batch['input_ids'][k], batch['labels'][k]
    shown = []
    for o, i, l in zip(orig, ids.tolist(), labels.tolist()):
        if l == -100:
            shown.append(o)
        else:
            new = tok.convert_ids_to_tokens(i)
            kind = 'MASK' if new == tok.mask_token else ('GIỮ' if new == o else 'RAND')
            shown.append(f'⟨{o}→{new}:{kind}⟩')
    print(f'Lần {k+1}:', ' '.join(shown), '\n')
print('Nhãn = -100 ở vị trí không được chọn → không tính loss.')

In [ ]:
# ============================================================
# B4. KV CACHE — ĐO TỐC ĐỘ SINH CÓ/KHÔNG CACHE
# ============================================================
from transformers import AutoModelForCausalLM, AutoTokenizer

gtok = AutoTokenizer.from_pretrained('distilgpt2')
gmod = AutoModelForCausalLM.from_pretrained('distilgpt2').to(device).eval()
inp = gtok('Deep learning for natural language processing', return_tensors='pt').to(device)

for use_cache in [True, False]:
    t0 = time.time()
    with torch.no_grad():
        gmod.generate(**inp, max_new_tokens=200, min_new_tokens=200, do_sample=False,
                      use_cache=use_cache, pad_token_id=gtok.eos_token_id)
    print(f'use_cache={use_cache!s:5}: {time.time()-t0:.2f}s cho 200 token')

def kv_bytes_per_token(L, n_kv, d_head, bytes_=2):
    return 2 * L * n_kv * d_head * bytes_
b = kv_bytes_per_token(28, 4, 128)
print(f'\nQwen2.5-7B: {b/1024:.0f} KB/token → 32K token ≈ {b*32768/1024**3:.2f} GB (fp16)')
print(f'Nếu không có GQA (28 KV heads): {kv_bytes_per_token(28, 28, 128)*32768/1024**3:.1f} GB')

In [ ]:
# ============================================================
# B5. LoRA MINI BẰNG PYTORCH (không cần GPU)
# ============================================================
import torch.nn as nn

class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B A x ;  W0 đóng băng, B khởi tạo 0."""
    def __init__(self, in_f, out_f, r=8, alpha=16):
        super().__init__()
        self.base = nn.Linear(in_f, out_f)
        self.base.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, in_f) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_f, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.scale
    @torch.no_grad()
    def merge(self):
        """Gộp BA vào W0 → suy luận không tốn thêm chi phí."""
        self.base.weight += self.scale * self.B @ self.A
        self.B.zero_()

def count(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)

full, lora = nn.Linear(768, 768), LoRALinear(768, 768, r=8)
print('Full  (tổng, trainable):', count(full))
print('LoRA  (tổng, trainable):', count(lora), f'→ {count(lora)[1]/count(full)[1]:.1%} tham số cần học')

# Kiểm tra: tại khởi tạo, LoRA = mô hình gốc; sau merge, đầu ra không đổi
x = torch.randn(4, 768)
print('Khởi tạo khớp gốc:', torch.allclose(lora(x), lora.base(x)))
opt = torch.optim.AdamW([p for p in lora.parameters() if p.requires_grad], lr=1e-2)
target = torch.randn(4, 768)
for _ in range(20):
    loss = F.mse_loss(lora(x), target); opt.zero_grad(); loss.backward(); opt.step()
before = lora(x).detach(); lora.merge(); after = lora(x).detach()
print('Merge giữ nguyên đầu ra:', torch.allclose(before, after, atol=1e-5))

In [ ]:
# ============================================================
# B6. DPO LOSS MINI + ĐỘ NHẠY THEO BETA
# ============================================================
def dpo_loss(pc, pr, rc, rr, beta=0.1):
    """pc/pr: log-prob chosen/rejected của policy; rc/rr: của reference."""
    logits = beta * ((pc - pr) - (rc - rr))
    return -F.logsigmoid(logits).mean(), logits

policy_chosen   = torch.tensor([-2.0, -1.5, -3.0])
policy_rejected = torch.tensor([-3.0, -2.0, -2.7])
ref_chosen      = torch.tensor([-2.2, -1.7, -3.1])
ref_rejected    = torch.tensor([-2.9, -2.1, -2.8])

for beta in [0.05, 0.1, 0.5, 1.0]:
    loss, logits = dpo_loss(policy_chosen, policy_rejected, ref_chosen, ref_rejected, beta)
    print(f'beta={beta:<4} loss={loss.item():.4f} | implicit reward margin = {logits.tolist()}')
print('\nNhận xét: mẫu thứ 3 policy đang ưa rejected hơn reference → margin âm → đóng góp loss lớn nhất.')
print('log(2)=0.693 là loss khi policy ≡ reference (margin = 0).')

# PHẦN C — ỨNG DỤNG XỬ LÝ RIÊNG CHO TIẾNG VIỆT 🇻🇳

> **⏱ Tóm tắt 30 giây.** Tiếng Việt viết cách theo **âm tiết**, không theo **từ** ("Hà Nội" = 1 từ, 2 âm tiết). Dấu thanh có nhiều cách mã hoá Unicode và nhiều kiểu đặt dấu. Mô hình tiếng Anh chia tiếng Việt thành rất nhiều token. → Ba việc bắt buộc: **chuẩn hoá Unicode/dấu**, **tách từ nếu mô hình yêu cầu** (PhoBERT), **chọn mô hình/tokenizer phù hợp**.

## C1. Đặc thù ngôn ngữ ảnh hưởng tới mô hình

| Đặc thù | Ví dụ | Hệ quả kỹ thuật |
|---|---|---|
| Khoảng trắng tách **âm tiết**, không tách từ | "học sinh", "trí tuệ nhân tạo" | Cần word segmentation (PhoBERT); nhãn NER phải theo từ |
| Đơn lập, không biến hình | "đi / đã đi / sẽ đi" | Thì, số nằm ở hư từ → hư từ quan trọng, **không nên bỏ stopword** máy móc |
| Dấu thanh + dấu mũ | a á à ả ã ạ â ấ … | Một ký tự = 1 hoặc 2–3 code point Unicode |
| **Unicode NFC vs NFD** | "ế" = U+1EBF hoặc e + U+0302 + U+0301 | Hai chuỗi *nhìn giống hệt* nhưng khác ID token → phải chuẩn hoá **NFC** |
| Hai kiểu đặt dấu | "hoà/hòa", "thuỷ/thủy", "xoè/xòe" | Cùng từ, khác token → chuẩn hoá về **một** kiểu |
| Văn bản không dấu, teencode | "ko bik", "dc", "j z" | Mạng xã hội; mô hình học trên báo chí xử lý kém |
| Trộn mã (code-switching) | "deadline gấp quá, check mail đi" | Cần tokenizer đa ngữ |
| Phương ngữ | "răng/sao", "mô/đâu", "heo/lợn" | Dữ liệu phân bố lệch về phương ngữ Bắc |

## C2. Bản đồ mô hình cho tiếng Việt (tham khảo, cập nhật thường xuyên trên HuggingFace)

| Nhóm | Mô hình | Ghi chú quan trọng |
|---|---|---|
| Encoder chuẩn | `vinai/phobert-base`, `-large`, `phobert-base-v2` | RoBERTa, **bắt buộc đầu vào đã tách từ** (VnCoreNLP), max **256** token; v2 train trên ~140GB |
| Encoder mạng xã hội | `uitnlp/visobert` | Train trên dữ liệu MXH, **không cần tách từ**, xử lý teencode/emoji tốt |
| Encoder RTD | ViDeBERTa | DeBERTaV3 cho tiếng Việt |
| Encoder đa ngữ | `FacebookAI/xlm-roberta-base/large`, mBERT | Không cần tách từ; baseline mạnh, dễ dùng |
| Seq2seq | `VietAI/vit5-base`, `vinai/bartpho-syllable`, `bartpho-word` | Tóm tắt, sinh tiêu đề, khôi phục dấu, sửa chính tả |
| Embedding (RAG) | `BAAI/bge-m3`, `intfloat/multilingual-e5-*`, `bkai-foundation-models/vietnamese-bi-encoder` | bkai cần đầu vào đã tách từ; e5 cần tiền tố `query:`/`passage:` |
| LLM | Qwen2.5/Qwen3, Llama 3.x, Gemma, SeaLLMs, Vistral | Kiểm tra trên benchmark tiếng Việt (VMLU) trước khi chọn |

## C3. Dữ liệu và benchmark tiếng Việt thường dùng

| Bài toán | Bộ dữ liệu |
|---|---|
| Cảm xúc / phản hồi | UIT-VSFC (phản hồi sinh viên), UIT-VSMEC (cảm xúc), ViGoEmotions |
| Ngôn từ thù ghét | ViHSD, ViHOS |
| NER | VLSP 2016/2018, PhoNER_COVID19 |
| QA đọc hiểu | UIT-ViQuAD 2.0 |
| Suy diễn ngôn ngữ | ViNLI |
| Đánh giá LLM | VMLU (trắc nghiệm đa lĩnh vực) |
| POS, cú pháp | VLSP, VnDT |

---

In [ ]:
# ============================================================
# C4. CHUẨN HOÁ TIẾNG VIỆT: UNICODE NFC + KIỂU ĐẶT DẤU + TEENCODE
# ============================================================
import unicodedata, re

TONE_MARKS = {'\u0301', '\u0300', '\u0309', '\u0303', '\u0323'}   # sắc, huyền, hỏi, ngã, nặng
VALID_PAIRS = {('o', 'a'), ('o', 'e'), ('u', 'y')}
_pat = re.compile(r'(?<![qQ])([oOuU])([^\W\d_])(?![^\W\d_])')

def normalize_tone_style(text: str) -> str:
    """Đưa về kiểu đặt dấu cũ (phổ biến trong corpus): hoá→hóa, thuỷ→thủy, xoè→xòe.
    Không đụng tới âm tiết có phụ âm cuối (hoán, khoảng) và 'qu' (quý)."""
    def repl(m):
        first, second = m.group(1), m.group(2)
        nfd = unicodedata.normalize('NFD', second)
        base, marks = nfd[0], nfd[1:]
        if (first.lower(), base.lower()) not in VALID_PAIRS:
            return m.group(0)
        if len(marks) != 1 or marks not in TONE_MARKS:
            return m.group(0)
        return unicodedata.normalize('NFC', first + marks) + base
    return _pat.sub(repl, text)

TEENCODE = {'ko': 'không', 'k0': 'không', 'khum': 'không', 'hok': 'không',
            'dc': 'được', 'đc': 'được', 'vs': 'với', 'mn': 'mọi người',
            'ntn': 'như thế nào', 'j': 'gì', 'bik': 'biết', 'thik': 'thích'}

def normalize_vi(text: str, fix_teencode=True, lower=False) -> str:
    text = unicodedata.normalize('NFC', text)          # 1) Unicode dựng sẵn
    text = normalize_tone_style(text)                   # 2) thống nhất kiểu dấu
    text = re.sub(r'(\w)\1{2,}', r'\1', text)           # 3) "hayyyy" → "hay"
    if fix_teencode:                                    # 4) teencode (từ điển nhỏ, cần mở rộng)
        text = re.sub(r'\b\w+\b', lambda m: TEENCODE.get(m.group(0).lower(), m.group(0)), text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower() if lower else text

def remove_accents(text: str) -> str:
    text = unicodedata.normalize('NFD', text)
    text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')
    return text.replace('đ', 'd').replace('Đ', 'D')

# --- Minh hoạ lỗi "giống mà khác" ---
a = 'Thuỷ lợi'                                   # NFC, kiểu mới
b = unicodedata.normalize('NFD', 'Thủy lợi')     # NFD, kiểu cũ
print('Hiển thị:', a, '|', b, '| bằng nhau?', a == b, '| số code point:', len(a), len(b))
print('Sau chuẩn hoá bằng nhau?', normalize_vi(a) == normalize_vi(b), '→', normalize_vi(a))

tests = ['Hoà bình', 'thuỷ điện', 'xoè ô', 'quý khách', 'khoảng cách', 'hoán đổi',
         'phim hayyyy mà giá vé ko rẻ, j z trời', 'Mọi người đc nghỉ ntn?']
for t in tests:
    print(f'{t:40s} → {normalize_vi(t)}')
print('\nBỏ dấu:', remove_accents('Trường Đại học Thủy lợi'))

In [ ]:
# ============================================================
# C5. TÁCH TỪ TIẾNG VIỆT (WORD SEGMENTATION)
# ============================================================
# PhoBERT được pre-train trên văn bản đã tách từ bằng VnCoreNLP (RDRSegmenter).
# Trên Colab, underthesea/pyvi dễ cài hơn (không cần Java); kết quả gần tương đương.
# Với nghiên cứu công bố: dùng đúng VnCoreNLP để khớp phân phối pre-training.

def word_segment(text: str) -> str:
    try:
        from underthesea import word_tokenize
        return word_tokenize(text, format='text')
    except Exception:
        from pyvi import ViTokenizer
        return ViTokenizer.tokenize(text)

sents = ['Trường Đại học Thủy lợi đào tạo kỹ sư tài nguyên nước.',
         'Ông Nguyễn Văn An làm việc tại Hà Nội từ năm 2020.',
         'Trí tuệ nhân tạo đang thay đổi ngành nông nghiệp Việt Nam.']
for s in sents:
    print(s, '\n  →', word_segment(normalize_vi(s)))

In [ ]:
# ============================================================
# C6. TOKENIZER FERTILITY — MÔ HÌNH NÀO "HIỂU" TIẾNG VIỆT RẺ NHẤT?
# ============================================================
# Fertility = số token / số âm tiết. Càng gần 1 càng tốt:
#  - ít token → context 512/2048 chứa được nhiều văn bản hơn
#  - LLM tính phí và thời gian theo token → rẻ và nhanh hơn
import pandas as pd

corpus = [
    'Biến đổi khí hậu làm gia tăng xâm nhập mặn ở Đồng bằng sông Cửu Long.',
    'Sinh viên cần nộp báo cáo thực hành trước thứ Sáu tuần sau.',
    'Hệ thống cảnh báo sạt lở bờ sông sử dụng cảm biến IoT và học sâu.',
    'Người dân phản ánh chất lượng nước sinh hoạt bị suy giảm nghiêm trọng.',
    'Mô hình ngôn ngữ lớn có thể trả lời câu hỏi bằng tiếng Việt khá tự nhiên.',
]
corpus = [normalize_vi(s) for s in corpus]
n_syll = sum(len(s.split()) for s in corpus)

tokenizers = {
    'GPT-2 (byte BPE, EN)':  ('openai-community/gpt2', False),
    'mBERT (WordPiece)':     ('bert-base-multilingual-cased', False),
    'XLM-R (SentencePiece)': ('FacebookAI/xlm-roberta-base', False),
    'PhoBERT (đã tách từ)':  ('vinai/phobert-base', True),
    'Qwen2.5 (byte BPE, đa ngữ)': ('Qwen/Qwen2.5-0.5B', False),
}
rows = []
for name, (ckpt, need_seg) in tokenizers.items():
    try:
        t = AutoTokenizer.from_pretrained(ckpt)
        texts = [word_segment(s) for s in corpus] if need_seg else corpus
        n_tok = sum(len(t.tokenize(s)) for s in texts)
        rows.append({'Tokenizer': name, 'Token': n_tok, 'Token/âm tiết': round(n_tok / n_syll, 2),
                     'Ví dụ': ' | '.join(t.tokenize(texts[0])[:12])})
    except Exception as e:
        rows.append({'Tokenizer': name, 'Token': None, 'Token/âm tiết': None, 'Ví dụ': f'Lỗi tải: {e}'[:60]})

pd.set_option('display.max_colwidth', 120)
display(pd.DataFrame(rows).sort_values('Token/âm tiết'))
print(f'Tổng số âm tiết: {n_syll}')

**Đọc kết quả (điển hình):** GPT-2 cần ~3–4 token/âm tiết (chẻ theo byte), mBERT/XLM-R ~1,2–1,5, PhoBERT **< 1** (một token cho cả từ ghép như `Đồng_bằng`), Qwen2.5 ~1,1–1,4.
→ Hệ quả thực tế: PhoBERT max 256 token vẫn chứa được đoạn văn dài hơn mBERT 512 token trong nhiều trường hợp; dùng GPT-2 cho tiếng Việt vừa tốn context vừa kém chất lượng.

In [ ]:
# ============================================================
# C7. PhoBERT FILL-MASK: CÓ TÁCH TỪ vs KHÔNG TÁCH TỪ
# ============================================================
from transformers import pipeline

pho_mlm = pipeline('fill-mask', model='vinai/phobert-base', top_k=5,
                   device=0 if torch.cuda.is_available() else -1)

pairs = [
    ('Hà Nội là <mask> của Việt Nam .',            'Hà_Nội là <mask> của Việt_Nam .'),
    ('Sinh viên đang học <mask> máy tại trường .',  'Sinh_viên đang học <mask> máy tại trường .'),
    ('Mùa lũ năm nay gây <mask> lở nghiêm trọng .', 'Mùa lũ năm nay gây <mask> lở nghiêm_trọng .'),
]
for raw, seg in pairs:
    print(f'\nKHÔNG tách từ: {raw}')
    print('   ', [(r["token_str"], round(r["score"], 3)) for r in pho_mlm(raw)])
    print(f'CÓ tách từ   : {seg}')
    print('   ', [(r["token_str"], round(r["score"], 3)) for r in pho_mlm(seg)])
print('\n→ Với đầu vào đúng định dạng pre-training (có "_"), dự đoán ổn định và hợp lý hơn.')

In [ ]:
# ============================================================
# C8. VĂN BẢN KHÔNG DẤU — MÔ HÌNH BỊ ẢNH HƯỞNG THẾ NÀO?
# ============================================================
with_acc = 'Sinh_viên đang học <mask> máy tại trường .'
no_acc   = 'Sinh vien dang hoc <mask> may tai truong .'
print('Có dấu   :', [r['token_str'] for r in pho_mlm(with_acc)])
print('Không dấu:', [r['token_str'] for r in pho_mlm(no_acc)])

pho_tok = AutoTokenizer.from_pretrained('vinai/phobert-base')
s = 'Người dân phản ánh chất lượng nước sinh hoạt bị suy giảm'
print('\nSố token có dấu   :', len(pho_tok.tokenize(word_segment(s))))
print('Số token không dấu:', len(pho_tok.tokenize(remove_accents(s))), pho_tok.tokenize(remove_accents(s))[:10])
print('\n→ Không dấu = mất thông tin + lệch phân phối. Giải pháp: khôi phục dấu (seq2seq ViT5/BARTpho)'
      ' hoặc dùng mô hình train trên MXH (ViSoBERT), hoặc augment dữ liệu bằng bản bỏ dấu.')

## C9. Fine-tune PhoBERT cho phân loại cảm xúc tiếng Việt

Pipeline chuẩn cho tiếng Việt: **chuẩn hoá → tách từ → tokenize (max 256) → fine-tune → phân tích lỗi**.
Cell dưới thử tải **UIT-VSFC** từ HuggingFace; nếu không tải được sẽ dùng bộ dữ liệu mini dựng sẵn để minh hoạ. Có tuỳ chọn `USE_LORA` để so sánh full fine-tuning với LoRA (PEFT).

In [ ]:
# ============================================================
# C9a. DỮ LIỆU
# ============================================================
from datasets import Dataset, DatasetDict, load_dataset

def load_vi_sentiment():
    try:
        ds = load_dataset('uitnlp/vietnamese_students_feedback')
        # nhãn gốc: 0=tiêu cực, 1=trung tính, 2=tích cực → giữ nhị phân cho gọn
        ds = ds.filter(lambda x: x['sentiment'] != 1)
        ds = ds.map(lambda x: {'text': x['sentence'], 'label': 1 if x['sentiment'] == 2 else 0})
        ds = DatasetDict({'train': ds['train'].shuffle(seed=SEED).select(range(3000)),
                          'test':  ds['test']})
        print('Đã tải UIT-VSFC (nhị phân, 3000 mẫu train).')
        return ds
    except Exception as e:
        print('Không tải được UIT-VSFC → dùng dữ liệu mini. Lý do:', str(e)[:80])
    pos = ['Thầy giảng bài rất dễ hiểu và nhiệt tình.', 'Môn học bổ ích, áp dụng được vào thực tế.',
           'Bài tập vừa sức, giúp em hiểu sâu kiến thức.', 'Cô luôn hỗ trợ sinh viên ngoài giờ học.',
           'Phòng thí nghiệm hiện đại, thiết bị đầy đủ.', 'Giáo trình rõ ràng, có nhiều ví dụ minh hoạ.',
           'Em rất thích cách thầy đặt câu hỏi gợi mở.', 'Đề thi sát nội dung, đánh giá công bằng.',
           'Lớp học sôi nổi, mọi người tích cực thảo luận.', 'Thầy phản hồi email rất nhanh.',
           'Môn này giúp em định hướng nghề nghiệp tốt hơn.', 'Slide đẹp, trình bày logic, dễ theo dõi.',
           'Cô chấm bài kỹ và góp ý chi tiết.', 'Thực hành trên Colab rất tiện, chạy được ngay.',
           'Nội dung cập nhật, bám sát xu hướng công nghệ.', 'Em học được rất nhiều từ đồ án cuối kỳ.']
    neg = ['Thầy giảng quá nhanh, em không theo kịp.', 'Bài tập quá nhiều so với số tín chỉ.',
           'Phòng học nóng, máy chiếu thường xuyên hỏng.', 'Giáo trình cũ, nhiều nội dung lỗi thời.',
           'Đề thi quá khó, không liên quan bài giảng.', 'Cô ít khi trả lời câu hỏi của sinh viên.',
           'Wifi yếu, không làm thực hành được.', 'Lịch học thay đổi liên tục, rất bất tiện.',
           'Slide sơ sài, chỉ đọc lại sách.', 'Điểm quá trình chấm không minh bạch.',
           'Môn học nhàm chán, không có ví dụ thực tế.', 'Máy tính phòng lab quá cũ, chạy rất chậm.',
           'Thầy hay đến muộn và cho nghỉ đột xuất.', 'Hướng dẫn đồ án không rõ ràng.',
           'Âm thanh trong giảng đường rất kém.', 'Không có tài liệu tham khảo bằng tiếng Việt.']
    texts, labels = pos + neg, [1]*len(pos) + [0]*len(neg)
    full = Dataset.from_dict({'text': texts, 'label': labels}).shuffle(seed=SEED)
    return full.train_test_split(test_size=0.25, seed=SEED)

raw = load_vi_sentiment()
print(raw)

In [ ]:
# ============================================================
# C9b. TIỀN XỬ LÝ + TOKENIZE + MODEL (full FT hoặc LoRA)
# ============================================================
from transformers import (AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding)
import evaluate

MODEL = 'vinai/phobert-base'      # thử thay: 'uitnlp/visobert' (không cần tách từ), 'FacebookAI/xlm-roberta-base'
NEED_SEG = 'phobert' in MODEL
USE_LORA = False                  # True → chỉ train adapter LoRA + head

tokenizer = AutoTokenizer.from_pretrained(MODEL)

def preprocess(batch):
    texts = [normalize_vi(t) for t in batch['text']]
    if NEED_SEG:
        texts = [word_segment(t) for t in texts]
    return tokenizer(texts, truncation=True, max_length=256)

tokenized = raw.map(preprocess, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=2, id2label={0: 'TIÊU_CỰC', 1: 'TÍCH_CỰC'}, label2id={'TIÊU_CỰC': 0, 'TÍCH_CỰC': 1})

if USE_LORA:
    from peft import LoraConfig, get_peft_model, TaskType
    cfg = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.1,
                     target_modules=['query', 'value'])
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
else:
    print(f'Full fine-tuning: {sum(p.numel() for p in model.parameters()):,} tham số trainable')

acc, f1 = evaluate.load('accuracy'), evaluate.load('f1')
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {'accuracy': acc.compute(predictions=preds, references=p.label_ids)['accuracy'],
            'macro_f1': f1.compute(predictions=preds, references=p.label_ids, average='macro')['f1']}

In [ ]:
# ============================================================
# C9c. HUẤN LUYỆN
# ============================================================
args = TrainingArguments(
    output_dir='./phobert_sentiment',
    num_train_epochs=4 if len(tokenized['train']) > 500 else 8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-4 if USE_LORA else 3e-5,   # LoRA thường cần LR lớn hơn
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    save_total_limit=1,
    logging_strategy='epoch',
    report_to='none',
    fp16=torch.cuda.is_available(),
    seed=SEED,
)
trainer = Trainer(model=model, args=args,
                  train_dataset=tokenized['train'], eval_dataset=tokenized['test'],
                  processing_class=tokenizer,
                  data_collator=DataCollatorWithPadding(tokenizer),
                  compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate())

In [ ]:
# ============================================================
# C9d. PHÂN TÍCH LỖI VỚI CÂU KHÓ: PHỦ ĐỊNH, MỈA MAI, TEENCODE, KHÔNG DẤU
# ============================================================
from sklearn.metrics import confusion_matrix, classification_report

pred = trainer.predict(tokenized['test'])
y_pred = pred.predictions.argmax(-1)
print(confusion_matrix(pred.label_ids, y_pred))
print(classification_report(pred.label_ids, y_pred, target_names=['TIÊU_CỰC', 'TÍCH_CỰC'], digits=3))

hard_cases = {
    'Phủ định kép':      'Không phải là môn học dở.',
    'Phủ định':          'Thầy không hề nhiệt tình chút nào.',
    'Mỉa mai':           'Hay quá, học xong chẳng nhớ được gì.',
    'Tương phản':        'Nội dung hay nhưng thầy giảng quá nhanh.',
    'Teencode':          'mon nay ok lắm mn ơi, thầy dạy dễ hỉu',
    'Không dấu':         'thay giang rat de hieu va nhiet tinh',
    'Trộn Anh–Việt':     'Deadline dày đặc, feedback thì chậm.',
}
model.eval()
for kind, text in hard_cases.items():
    x = normalize_vi(text)
    x = word_segment(x) if NEED_SEG else x
    enc = tokenizer(x, return_tensors='pt', truncation=True, max_length=256).to(model.device)
    with torch.no_grad():
        prob = model(**enc).logits.softmax(-1)[0]
    lab = model.config.id2label[int(prob.argmax())]
    print(f'{kind:15s} | {text:45s} → {lab:9s} ({prob.max():.2f})')
print('\nCâu hỏi: lỗi nào do dữ liệu (thiếu mẫu), lỗi nào do mô hình (PhoBERT học trên văn bản chuẩn)?'
      '\nThử lại với MODEL = "uitnlp/visobert" và so sánh.')

## C10. NER tiếng Việt — căn nhãn khi tokenizer không có `word_ids()`

Tokenizer của PhoBERT là tokenizer "chậm" (không có `word_ids()`), nên ta **tự căn nhãn**: tokenize từng từ, gán nhãn cho subword đầu, subword sau gán −100.
Lưu ý: với PhoBERT, đơn vị là **từ đã tách** (`Nguyễn_Văn_An` là 1 từ) → nhãn BIO cũng phải theo từ đã tách. Bộ dữ liệu PhoNER_COVID19 cung cấp sẵn cả bản âm tiết và bản đã tách từ.

In [ ]:
# ============================================================
# C10. CĂN NHÃN BIO CHO PhoBERT + VÒNG HUẤN LUYỆN RÚT GỌN
# ============================================================
from transformers import AutoModelForTokenClassification

labels_list = ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']
l2i = {l: i for i, l in enumerate(labels_list)}

# Câu đã tách từ + nhãn theo từ (định dạng giống PhoNER_COVID19 bản word-level)
ner_data = [
    (['Ông', 'Nguyễn_Văn_An', 'làm_việc', 'tại', 'Trường', 'Đại_học', 'Thủy_lợi', '.'],
     ['O', 'B-PER', 'O', 'O', 'B-ORG', 'I-ORG', 'I-ORG', 'O']),
    (['Bà', 'Trần_Thị_Bình', 'sống', 'ở', 'Hà_Nội', '.'],
     ['O', 'B-PER', 'O', 'O', 'B-LOC', 'O']),
    (['Công_ty', 'VinAI', 'có', 'văn_phòng', 'tại', 'Thành_phố', 'Hồ_Chí_Minh', '.'],
     ['B-ORG', 'I-ORG', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'O']),
    (['Lũ', 'lớn', 'xảy_ra', 'ở', 'Quảng_Nam', 'và', 'Đà_Nẵng', '.'],
     ['O', 'O', 'O', 'O', 'B-LOC', 'O', 'B-LOC', 'O']),
]

ner_tok = AutoTokenizer.from_pretrained('vinai/phobert-base')

def encode_words(words, tags, max_len=256):
    ids, lab = [ner_tok.cls_token_id], [-100]
    for w, t in zip(words, tags):
        piece_ids = ner_tok.encode(w, add_special_tokens=False) or [ner_tok.unk_token_id]
        ids += piece_ids
        lab += [l2i[t]] + [-100] * (len(piece_ids) - 1)   # chỉ subword đầu có nhãn
    ids, lab = ids[:max_len-1] + [ner_tok.sep_token_id], lab[:max_len-1] + [-100]
    return {'input_ids': ids, 'attention_mask': [1]*len(ids), 'labels': lab}

ex = encode_words(*ner_data[2])
for tid, l in zip(ex['input_ids'], ex['labels']):
    print(f'{ner_tok.convert_ids_to_tokens(tid):15s} {labels_list[l] if l != -100 else "-100"}')

# Huấn luyện vài bước để thấy loss giảm (dữ liệu thật: PhoNER_COVID19 / VLSP 2016)
ner_model = AutoModelForTokenClassification.from_pretrained(
    'vinai/phobert-base', num_labels=len(labels_list),
    id2label=dict(enumerate(labels_list)), label2id=l2i).to(device)
from transformers import DataCollatorForTokenClassification
coll = DataCollatorForTokenClassification(ner_tok)
batch = coll([encode_words(w, t) for w, t in ner_data]).to(device)
opt = torch.optim.AdamW(ner_model.parameters(), lr=5e-5)
ner_model.train()
for step in range(30):
    out = ner_model(**batch); opt.zero_grad(); out.loss.backward(); opt.step()
    if step % 10 == 0 or step == 29:
        print(f'step {step:2d} loss={out.loss.item():.4f}')

## C11. RAG tiếng Việt: BM25 + dense embedding + RRF

**Vì sao lai (hybrid)?** BM25 bắt tốt **từ khoá chính xác** (mã học phần, tên riêng, số hiệu văn bản); dense embedding bắt **đồng nghĩa/diễn đạt khác** ("sạt lở" ≈ "xói lở"). Ghép bằng **Reciprocal Rank Fusion**:
$$\text{RRF}(d)=\sum_{r\in\{\text{BM25},\text{dense}\}}\frac{1}{k+\text{rank}_r(d)},\quad k=60$$

Lưu ý riêng cho tiếng Việt:
- BM25 trên **từ đã tách** tốt hơn trên âm tiết ("học" trong "học sinh" ≠ "học" trong "học tập").
- Chunk theo câu/đoạn; tránh cắt giữa từ ghép; giữ tiêu đề mục trong mỗi chunk.
- Mô hình e5 cần tiền tố `query:` / `passage:`; `bkai-foundation-models/vietnamese-bi-encoder` cần đầu vào đã tách từ.

In [ ]:
# ============================================================
# C11a. KHO TRI THỨC + TRUY HỒI LAI
# ============================================================
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

docs = [
    'PhoBERT là mô hình ngôn ngữ tiền huấn luyện cho tiếng Việt do VinAI phát triển, dựa trên RoBERTa và yêu cầu văn bản đầu vào đã được tách từ.',
    'ViSoBERT được huấn luyện trên dữ liệu mạng xã hội tiếng Việt, xử lý tốt teencode, emoji và văn bản không chuẩn.',
    'Unicode có hai dạng biểu diễn chữ có dấu là NFC (dựng sẵn) và NFD (tổ hợp); cần chuẩn hoá về NFC trước khi đưa vào mô hình.',
    'LoRA là kỹ thuật tinh chỉnh tiết kiệm tham số, chỉ học hai ma trận hạng thấp và giữ nguyên trọng số gốc.',
    'Hệ thống giám sát xói lở bờ sông dùng cảm biến IoT đo độ nghiêng, độ ẩm đất và mực nước để cảnh báo sớm.',
    'Xâm nhập mặn ở Đồng bằng sông Cửu Long ngày càng nghiêm trọng do biến đổi khí hậu và suy giảm dòng chảy thượng nguồn.',
    'RAG kết hợp truy hồi tài liệu với mô hình sinh để giảm hiện tượng bịa đặt thông tin của mô hình ngôn ngữ lớn.',
    'BM25 là thuật toán truy hồi dựa trên tần suất từ khoá, hiệu quả với tên riêng và mã số chính xác.',
]
docs_norm = [normalize_vi(d) for d in docs]
docs_seg  = [word_segment(d).lower().split() for d in docs_norm]
bm25 = BM25Okapi(docs_seg)

emb_model = SentenceTransformer('intfloat/multilingual-e5-small', device=device)
doc_emb = emb_model.encode(['passage: ' + d for d in docs_norm], normalize_embeddings=True)

def hybrid_search(query, k=3, k_rrf=60):
    q = normalize_vi(query)
    bm_scores = bm25.get_scores(word_segment(q).lower().split())
    dn_scores = doc_emb @ emb_model.encode(['query: ' + q], normalize_embeddings=True)[0]
    bm_rank = {i: r for r, i in enumerate(np.argsort(-bm_scores))}
    dn_rank = {i: r for r, i in enumerate(np.argsort(-dn_scores))}
    fused = {i: 1/(k_rrf + bm_rank[i] + 1) + 1/(k_rrf + dn_rank[i] + 1) for i in range(len(docs))}
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [(i, fused[i], bm_rank[i] + 1, dn_rank[i] + 1) for i in top]

for q in ['Mô hình nào hợp với bình luận Facebook có teencode?',
          'Làm sao phát hiện sạt lở bờ sông sớm?',
          'Vì sao chữ có dấu nhìn giống nhau mà máy lại coi là khác?']:
    print('\nHỏi:', q)
    for i, s, rb, rd in hybrid_search(q):
        print(f'  [{i}] RRF={s:.4f} (BM25 #{rb}, dense #{rd}) {docs[i][:85]}…')

In [ ]:
# ============================================================
# C11b. SINH CÂU TRẢ LỜI CÓ TRÍCH DẪN VỚI LLM NHỎ (tuỳ chọn, cần GPU)
# ============================================================
USE_LLM = torch.cuda.is_available()
LLM_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'   # ~3GB fp16, chạy tốt trên T4

def build_prompt(question, hits):
    ctx = '\n'.join(f'[{i}] {docs[i]}' for i, *_ in hits)
    return [
        {'role': 'system', 'content': 'Bạn là trợ giảng NLP. Chỉ trả lời dựa trên tài liệu được cung cấp, '
                                      'trích dẫn số hiệu [i] sau mỗi ý. Nếu tài liệu không có thông tin, nói "Không tìm thấy trong tài liệu".'},
        {'role': 'user', 'content': f'Tài liệu:\n{ctx}\n\nCâu hỏi: {question}'},
    ]

def answer_with_context(question, k=3):
    hits = hybrid_search(question, k=k)
    msgs = build_prompt(question, hits)
    if not USE_LLM:
        return 'Không có GPU — trả về ngữ cảnh truy hồi:\n' + '\n'.join(f'[{i}] {docs[i]}' for i, *_ in hits)
    ids = llm_tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(device)
    with torch.no_grad():
        out = llm.generate(ids, max_new_tokens=200, do_sample=False, repetition_penalty=1.05)
    return llm_tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

if USE_LLM:
    llm_tok = AutoTokenizer.from_pretrained(LLM_NAME)
    llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16).to(device).eval()

for q in ['PhoBERT cần tiền xử lý gì đặc biệt?', 'Giá cổ phiếu VinAI hôm nay bao nhiêu?']:
    print('Hỏi:', q, '\nĐáp:', answer_with_context(q), '\n' + '-'*70)

## C12. Checklist chọn giải pháp cho bài toán tiếng Việt

| Bài toán | Lựa chọn đầu tiên | Tiền xử lý bắt buộc | Chỉ số báo cáo |
|---|---|---|---|
| Phân loại văn bản chuẩn (báo, văn bản hành chính) | PhoBERT-base-v2 | NFC + tách từ, max 256 | Macro-F1, ≥3 seed |
| Bình luận mạng xã hội, teencode | ViSoBERT / XLM-R | NFC, giữ emoji | Macro-F1 theo lớp |
| NER | PhoBERT + căn nhãn subword | Nhãn theo từ đã tách | Entity-level F1 (seqeval) |
| Tóm tắt, sinh tiêu đề, khôi phục dấu | ViT5 / BARTpho | NFC | ROUGE, BERTScore + đánh giá người |
| Hỏi đáp trên tài liệu nội bộ | RAG: BGE-M3/e5 + BM25 + LLM đa ngữ | Chunk theo đoạn, giữ tiêu đề | Recall@k, faithfulness |
| Chatbot, agent | LLM đa ngữ (Qwen/Llama/Gemma) + RAG + LoRA nếu cần giọng điệu riêng | Chat template đúng | Đánh giá người + bộ test riêng |

**Năm lỗi hay gặp:** (1) quên chuẩn hoá NFC; (2) đưa văn bản chưa tách từ vào PhoBERT; (3) để PhoBERT nhận > 256 token → lỗi index; (4) bỏ stopword làm mất phủ định "không/chưa/chẳng"; (5) chỉ báo accuracy trên dữ liệu mất cân bằng.

---

# PHẦN D — THỰC HÀNH GỐC (ĐÃ CẬP NHẬT API, CHẠY ĐƯỢC TRỰC TIẾP)

In [ ]:
# ============================================================
# D1. MLM VỚI mBERT/DistilBERT ĐA NGỮ (so sánh với PhoBERT ở C7)
# ============================================================
mlm = pipeline('fill-mask', model='distilbert-base-multilingual-cased', top_k=5,
               device=0 if torch.cuda.is_available() else -1)
for s in ['Hôm nay trời [MASK] rất đẹp và dễ chịu.',
          'Sinh viên học [MASK] sâu trong môn xử lý ngôn ngữ tự nhiên.',
          'BERT là mô hình dùng cơ chế [MASK] để biểu diễn ngữ cảnh hai chiều.']:
    print('\nCâu:', s)
    for r in mlm(s):
        print(f"  {r['token_str']:15s} score={r['score']:.4f}")
print('\nNhận xét: mBERT thường đoán "mảnh" âm tiết hoặc từ tiếng Anh — so với PhoBERT ở C7.')

In [ ]:
# ============================================================
# D2. GPT-2: SO SÁNH CHIẾN LƯỢC DECODING + PHÂN PHỐI TOKEN KẾ TIẾP
# ============================================================
gpt_tok = AutoTokenizer.from_pretrained('openai-community/gpt2')
gpt = AutoModelForCausalLM.from_pretrained('openai-community/gpt2').to(device).eval()
PROMPT = 'Artificial intelligence will change the world by'
inputs = gpt_tok(PROMPT, return_tensors='pt').to(device)

strategies = {
    'Greedy':            dict(do_sample=False),
    'Beam (k=5)':        dict(num_beams=5, early_stopping=True, no_repeat_ngram_size=2),
    'Temperature 0.7':   dict(do_sample=True, temperature=0.7, top_k=0),
    'Temperature 1.5':   dict(do_sample=True, temperature=1.5, top_k=0),
    'Top-k (40)':        dict(do_sample=True, top_k=40),
    'Top-p (0.9)':       dict(do_sample=True, top_p=0.9, top_k=0),
    'Min-p (0.1), T=1.5':dict(do_sample=True, min_p=0.1, temperature=1.5, top_k=0),
}
torch.manual_seed(SEED)
for name, kw in strategies.items():
    try:
        with torch.no_grad():
            out = gpt.generate(**inputs, max_new_tokens=40, pad_token_id=gpt_tok.eos_token_id, **kw)
        print(f'\n[{name}] {gpt_tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()}')
    except Exception as e:
        print(f'\n[{name}] không hỗ trợ trong phiên bản này: {e}')

def next_token_dist(prompt, top_n=8):
    enc = gpt_tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        probs = gpt(**enc).logits[0, -1].softmax(-1)
    p, ids = probs.topk(top_n)
    H = -(probs * probs.clamp_min(1e-12).log()).sum().item()   # entropy trên TOÀN vocab
    print(f'\n"{prompt}"  | entropy = {H:.2f} nats')
    for pi, ii in zip(p, ids):
        print(f'   {gpt_tok.decode([int(ii)])!r:14s} {pi.item():.3f} ' + '█' * int(pi.item() * 60))

for p in ['The capital of France is', 'She went to the']:
    next_token_dist(p)

In [ ]:
# ============================================================
# D3. PERPLEXITY — VÀ CÁI BẪY KHI ĐO TIẾNG VIỆT BẰNG MÔ HÌNH TIẾNG ANH
# ============================================================
def perplexity(text, model, tok):
    enc = tok(text, return_tensors='pt').to(device)
    with torch.no_grad():
        loss = model(**enc, labels=enc['input_ids']).loss
    return math.exp(loss.item()), enc['input_ids'].shape[1]

for t in ['The student studies natural language processing at university.',
          'University processing language natural at studies student the.',
          'Sinh viên học xử lý ngôn ngữ tự nhiên ở trường đại học.',
          'Trường đại học ở nhiên tự ngữ ngôn lý xử học viên sinh.']:
    ppl, n = perplexity(t, gpt, gpt_tok)
    print(f'PPL={ppl:9.1f} | {n:3d} token | {t}')
print('\nCâu tiếng Việt bị chẻ thành rất nhiều byte-token, nhiều token "dễ đoán" (mảnh của cùng một ký tự)'
      '\n→ PPL/token không phản ánh đúng mức hiểu. Không so PPL giữa các tokenizer khác nhau.')

In [ ]:
# ============================================================
# D4. EMBEDDING NGỮ CẢNH: [CLS] THÔ vs SENTENCE EMBEDDING
# ============================================================
from transformers import AutoModel
enc_tok = AutoTokenizer.from_pretrained('distilbert-base-multilingual-cased')
enc_model = AutoModel.from_pretrained('distilbert-base-multilingual-cased').to(device).eval()

sentences = ['Tôi ra ngân hàng rút tiền.', 'Tôi gửi tiết kiệm ở ngân hàng.',
             'Chúng tôi ngồi câu cá bên bờ sông.', 'Nước sông dâng cao làm sạt lở bờ.']

def cls_emb(s):
    x = enc_tok(s, return_tensors='pt').to(device)
    with torch.no_grad():
        return enc_model(**x).last_hidden_state[0, 0].cpu()

cls = torch.stack([cls_emb(s) for s in sentences])
sbert = torch.tensor(emb_model.encode(['query: ' + s for s in sentences], normalize_embeddings=True))

def sim_table(E, title):
    E = F.normalize(E, dim=-1); S = (E @ E.T).numpy()
    print(f'\n{title}'); print(pd.DataFrame(S.round(3), index=[s[:22] for s in sentences], columns=range(4)))
sim_table(cls, '[CLS] của DistilBERT đa ngữ (chưa huấn luyện cho similarity)')
sim_table(sbert, 'multilingual-e5 (huấn luyện contrastive)')
print('\n→ [CLS] thô cho similarity cao "đều đều" giữa mọi câu (anisotropy); embedding contrastive tách nhóm rõ hơn.')

# PHẦN E — THẢO LUẬN, BÀI TẬP, TÀI LIỆU

## XII. CÂU HỎI THẢO LUẬN TRÊN LỚP

**Câu 1 — MLM 80/10/10.** Phân tích vai trò từng thành phần. Thiết kế thí nghiệm so sánh 100%-[MASK] với 80/10/10 (gợi ý: pre-train tiếp một BERT nhỏ vài nghìn bước trên corpus tiếng Việt, đo F1 downstream).

**Câu 2 — Bidirectionality.** Vì sao không dùng BERT trực tiếp để sinh văn bản? Cần thay đổi gì (mask, mục tiêu, cách giải mã)? So sánh với GPT-2 và với BART.

**Câu 3 — In-Context Learning.** ICL có phải "học" không? Vì sao thêm ví dụ lại tốt hơn dù không có gradient?

**Câu 4 — Hallucination.** Mục tiêu CLM có thưởng/phạt tính đúng sự thật không? RAG và DPO giảm hallucination theo cơ chế khác nhau thế nào? Vì sao không loại bỏ hoàn toàn được?

**Câu 5 — Bias.** Mô hình BERT sàng lọc CV: liệt kê ≥3 nguồn bias (pre-training, fine-tuning, gán nhãn), 2 cách phát hiện, 2 cách giảm thiểu.

**Câu 6 — 🇻🇳 Tách từ.** Khi nào tách từ giúp ích (PhoBERT, BM25) và khi nào không cần/không nên (ViSoBERT, XLM-R, LLM)? Lỗi tách từ lan truyền sang NER như thế nào?

**Câu 7 — 🇻🇳 Tokenizer và chi phí.** Một LLM tính phí theo token. Với cùng một văn bản tiếng Việt, tokenizer A cho fertility 3,5, tokenizer B cho 1,2. Chi phí và độ dài ngữ cảnh hiệu dụng chênh lệch bao nhiêu? Điều này có ảnh hưởng tới công bằng giữa các ngôn ngữ không?

---

## XIII. BÀI TẬP VỀ NHÀ

### 13.1 Lý thuyết

**Bài 1 (2đ) — Toán MLM.** Câu "Hôm nay **học** sinh đến **trường** rất đúng **giờ**." với 3 token được chọn.
(a) Viết 3 phiên bản đầu vào có thể theo 80/10/10. (b) Viết $\mathcal L_{\text{MLM}}$ cụ thể. (c) Nếu tính loss ở 100% token (kể cả token không bị che), mục tiêu trở thành gì và vì sao tệ? (d) MLM cần 40 epoch, CLM cần 10 epoch để hội tụ: so sánh tổng số dự đoán. (e) 🇻🇳 Với PhoBERT, "học_sinh" là một token — việc che "học" riêng lẻ còn ý nghĩa không? Đề xuất whole-word masking.

**Bài 2 (2đ) — Kiến trúc.** (a) Tính số tham số 1 layer và toàn bộ BERT-Base, GPT-2 Medium (dùng công thức ở 4.2, kiểm chứng bằng cell B1). (b) Vì sao GPT không cần segment embedding; hạn chế gì với NLI? (c) Vì sao [CLS] thành biểu diễn tổng hợp. (d) Tính bộ nhớ KV cache của một LLM 32 layer, 8 KV head, $d_{head}$=128, context 16K, fp16.

**Bài 3 (1đ) — Đọc paper.** Devlin et al. (2019), Sections 1, 3, 4: feature-based vs fine-tuning; nguồn gốc con số 340M; mức cải thiện GLUE và task cải thiện nhiều nhất.

### 13.2 Lập trình

**Bài 4 (3đ) — PhoBERT cho tiếng Việt.** (a) NER trên PhoNER_COVID19 (bản word-level), báo cáo F1 theo từng loại thực thể bằng `seqeval`. (b) So sánh PhoBERT vs XLM-R vs mBERT: F1, thời gian/epoch, thời gian suy luận/mẫu, số tham số, fertility. (c) Trực quan hoá attention trên "Anh Nam đến trường. Anh ấy học rất chăm chỉ." — "ấy" có chú ý tới "Nam" không, ở layer/head nào?

**Bài 5 (2đ) — GPT-2 và perplexity.** (a) PPL của 10 câu tự nhiên vs 10 câu vô nghĩa, vẽ histogram. (b) Với "The future of AI is", sinh ở $T\in\{0.3,0.7,1.0,1.3,1.8\}$; vẽ $T$–PPL và $T$–distinct-2; nhận xét trade-off.

### 13.3 Bài tập bổ sung (trung bình → nâng cao)

**BT1 — Trung bình: MLM vs CLM trên tiếng Việt.** 10 câu chủ đề giáo dục/công nghệ; top-3 dự đoán của mBERT và PhoBERT (có tách từ); PPL câu gốc vs câu đảo bằng `Qwen/Qwen2.5-0.5B`. Nộp bảng kết quả + nhận xét.

**BT2 — Khá: Phân loại cảm xúc tiếng Việt.** ≥100 câu (hoặc UIT-VSFC); fine-tune PhoBERT và ViSoBERT; báo cáo accuracy, macro-F1, confusion matrix trên 3 seed; phân tích lỗi trên nhóm câu phủ định, mỉa mai, teencode, không dấu.

**BT3 — Nâng cao: Mini RAG tài liệu môn học.** ≥20 đoạn về BERT, GPT, LoRA, RAG, tokenization; 3 chiến lược chunking (câu, đoạn, cửa sổ trượt); so sánh BM25 âm tiết / BM25 từ đã tách / dense / hybrid bằng Recall@k trên ≥15 câu hỏi có đáp án; hàm `answer_with_context` có trích dẫn.

**BT4 — Nâng cao: LoRA/DPO mini.** So sánh full FT vs LoRA ($r\in\{4,8,16\}$) trên PhoBERT; tạo 20 cặp (prompt, chosen, rejected) cho trợ giảng NLP tiếng Việt; cài DPO loss trên log-prob thật của một causal LM nhỏ; khảo sát $\beta\in\{0.05,0.1,0.5,1.0\}$.

**BT5 — 🇻🇳 Nâng cao: Khôi phục dấu tiếng Việt.** Tạo cặp (không dấu → có dấu) từ văn bản có sẵn bằng `remove_accents`; fine-tune `VietAI/vit5-base` hoặc `vinai/bartpho-syllable`; đo độ chính xác theo âm tiết; phân tích lỗi trên từ đồng dạng không dấu ("ma/má/mà/mã/mạ").

**BT6 — 🇻🇳 Nghiên cứu nhỏ: Ảnh hưởng của chuẩn hoá.** Trên một bộ dữ liệu MXH (ví dụ ViHSD), so sánh 4 cấu hình: không chuẩn hoá / NFC / NFC + kiểu dấu / NFC + kiểu dấu + teencode. Báo cáo macro-F1 trung bình 3 seed và kiểm định ý nghĩa thống kê.

### 13.4 Câu hỏi mở
1. **ELECTRA vs BERT:** vì sao RTD hiệu quả mẫu hơn MLM; vì sao ELECTRA-Small cạnh tranh được với mô hình lớn hơn nhiều; nhược điểm?
2. **RLHF, DPO và Constitutional AI:** các giai đoạn RLHF; Constitutional AI (Anthropic, 2022) dùng phản hồi từ AI theo bộ nguyên tắc để giảm nhu cầu nhãn người như thế nào; reward hacking là gì và giảm thiểu ra sao?

---

## XIV. TÀI LIỆU THAM KHẢO

### 14.1 Paper nền tảng
1. Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL-HLT*, 4171–4186.
2. Radford, A., Narasimhan, K., Salimans, T., & Sutskever, I. (2018). Improving language understanding by generative pre-training. *OpenAI*.
3. Radford, A., Wu, J., Child, R., Luan, D., Amodei, D., & Sutskever, I. (2019). Language models are unsupervised multitask learners. *OpenAI*.
4. Brown, T. B., et al. (2020). Language models are few-shot learners. *NeurIPS*.
5. Liu, Y., et al. (2019). RoBERTa: A robustly optimized BERT pretraining approach. *arXiv:1907.11692*.
6. Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT. *NeurIPS EMC² Workshop*.
7. Clark, K., Luong, M.-T., Le, Q. V., & Manning, C. D. (2020). ELECTRA. *ICLR*.
8. Raffel, C., et al. (2020). Exploring the limits of transfer learning with a unified text-to-text transformer (T5). *JMLR*.
9. Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2020). The curious case of neural text degeneration. *ICLR*.
10. Kaplan, J., et al. (2020). Scaling laws for neural language models. *arXiv:2001.08361*.
11. Hoffmann, J., et al. (2022). Training compute-optimal large language models (Chinchilla). *NeurIPS*.
12. Ouyang, L., et al. (2022). Training language models to follow instructions with human feedback. *NeurIPS*.
13. Hu, E. J., et al. (2022). LoRA: Low-rank adaptation of large language models. *ICLR*.
14. Dettmers, T., et al. (2023). QLoRA: Efficient finetuning of quantized LLMs. *NeurIPS*.
15. Rafailov, R., et al. (2023). Direct preference optimization. *NeurIPS*. arXiv:2305.18290.
16. Bai, Y., et al. (2022). Constitutional AI: Harmlessness from AI feedback. *arXiv:2212.08073*.
17. Warner, B., et al. (2024). Smarter, better, faster, longer: A modern bidirectional encoder (ModernBERT). *arXiv:2412.13663*.
18. Reimers, N., & Gurevych, I. (2019). Sentence-BERT. *EMNLP*.
19. Liu, N. F., et al. (2024). Lost in the middle: How language models use long contexts. *TACL*.

### 14.2 🇻🇳 NLP tiếng Việt
20. Nguyen, D. Q., & Nguyen, A. T. (2020). PhoBERT: Pre-trained language models for Vietnamese. *Findings of EMNLP*.
21. Nguyen, Q.-N., et al. (2023). ViSoBERT: A pre-trained language model for Vietnamese social media text processing. *EMNLP*.
22. Tran, C. D., et al. (2023). ViDeBERTa: A powerful pre-trained language model for Vietnamese. *Findings of EACL*.
23. Phan, L., et al. (2022). ViT5: Pretrained text-to-text transformer for Vietnamese language generation. *NAACL SRW*.
24. Tran, N. L., Le, D. M., & Nguyen, D. Q. (2022). BARTpho: Pre-trained sequence-to-sequence models for Vietnamese. *Interspeech*.
25. Vu, T., et al. (2018). VnCoreNLP: A Vietnamese natural language processing toolkit. *NAACL Demonstrations*.
26. Truong, T. H., Dao, M. H., & Nguyen, D. Q. (2021). COVID-19 named entity recognition for Vietnamese (PhoNER_COVID19). *NAACL*.
27. Nguyen, K. V., et al. (2018). UIT-VSFC: Vietnamese students' feedback corpus for sentiment analysis. *KSE*.
28. Luu, S. T., Nguyen, K. V., & Nguyen, N. L.-T. (2021). A large-scale dataset for hate speech detection on Vietnamese social media texts (ViHSD). *IEA/AIE*.
29. Nguyen, K. V., et al. (2020). A Vietnamese dataset for evaluating machine reading comprehension (UIT-ViQuAD). *COLING*.

### 14.3 Sách và khoá học
- Jurafsky & Martin, *Speech and Language Processing* (3rd ed. draft) — web.stanford.edu/~jurafsky/slp3
- Zhang, Lipton, Li & Smola, *Dive into Deep Learning* — d2l.ai (chương NLP Pretraining)
- Tunstall, von Werra & Wolf (2022), *Natural Language Processing with Transformers*, O'Reilly
- Jay Alammar, *The Illustrated BERT* — jalammar.github.io/illustrated-bert
- Stanford CS224N (Pretraining lecture); HuggingFace LLM Course — huggingface.co/learn

### 14.4 Tài liệu kỹ thuật
- HF Transformers training: https://huggingface.co/docs/transformers/training
- HF PEFT / LoRA: https://huggingface.co/docs/peft/index
- ModernBERT: https://huggingface.co/blog/modernbert
- HF Advanced RAG cookbook: https://huggingface.co/learn/cookbook/advanced_rag
- VinAI Research: https://github.com/VinAIResearch · UIT NLP: https://huggingface.co/uitnlp

---
*Bài giảng Buổi 7 — Kết thúc.* © Tài liệu giảng dạy học phần "Deep Learning trong NLP". Vui lòng không phân phối ngoài phạm vi lớp học khi chưa có sự đồng ý của giảng viên.